In [ ]:
!pip install -U transformers accelerate datasets scikit-learn pandas numpy openpyxl matplotlib seaborn

In [ ]:
import os
import re
import random
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoConfig
)

In [ ]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH = "/content/drive/MyDrive/reviews_dataset.xlsx"

POSITIVE_PATH = "/content/positive_reviews.csv"
NEUTRAL_PATH = "/content/neutral_reviews.csv"
NEGATIVE_PATH = "/content/negative_reviews.csv"


def normalize_column_name(name):
    return re.sub(r"[^a-zа-яё0-9]+", "", str(name).strip().lower())


def find_column(df, candidates):
    normalized_map = {
        normalize_column_name(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize_column_name(candidate)

        if key in normalized_map:
            return normalized_map[key]

    return None


def read_table(path):
    if path.endswith(".xlsx"):
        return pd.read_excel(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError("Поддерживаются только .xlsx и .csv")


def load_dataset_from_rating_file(path):
    raw_df = read_table(path)

    text_col = find_column(
        raw_df,
        [
            "text", "review_text", "review", "comment",
            "отзыв", "отзывы", "текст", "текст отзыва", "комментарий"
        ]
    )

    rating_col = find_column(
        raw_df,
        [
            "rating", "оценка", "рейтинг", "stars", "звезды", "звёзды"
        ]
    )

    if text_col is None:
        raise ValueError("Не найдена колонка с текстом отзыва.")

    if rating_col is None:
        raise ValueError("Не найдена колонка с рейтингом.")

    data = raw_df[[text_col, rating_col]].copy()
    data.columns = ["text", "rating"]

    data["text"] = data["text"].fillna("").astype(str).str.strip()
    data["rating"] = pd.to_numeric(data["rating"], errors="coerce")

    data = data.dropna(subset=["rating"])
    data["rating"] = data["rating"].astype(int)
    data = data[data["rating"].isin([1, 2, 3, 4, 5])]
    data = data[data["text"].str.len() >= 20]
    data = data.drop_duplicates(subset=["text"])

    def rating_to_label(rating):
        if rating == 3:
            return 0  # neutral
        if rating in [1, 2]:
            return 1  # negative
        if rating in [4, 5]:
            return 2  # positive
        return None

    data["label"] = data["rating"].apply(rating_to_label)
    data = data.dropna(subset=["label"])
    data["label"] = data["label"].astype(int)

    return data[["text", "label"]].reset_index(drop=True)


def load_dataset_from_three_files(positive_path, neutral_path, negative_path):
    if not os.path.exists(positive_path):
        raise FileNotFoundError(f"Не найден файл: {positive_path}")

    if not os.path.exists(neutral_path):
        raise FileNotFoundError(f"Не найден файл: {neutral_path}")

    if not os.path.exists(negative_path):
        raise FileNotFoundError(f"Не найден файл: {negative_path}")

    df_pos = pd.read_csv(positive_path)
    df_neu = pd.read_csv(neutral_path)
    df_neg = pd.read_csv(negative_path)

    for name, part in [
        ("positive", df_pos),
        ("neutral", df_neu),
        ("negative", df_neg)
    ]:
        text_col = find_column(
            part,
            [
                "text", "review_text", "review", "comment",
                "отзыв", "отзывы", "текст", "текст отзыва", "комментарий"
            ]
        )

        if text_col is None:
            raise ValueError(f"В файле {name} не найдена колонка с текстом.")

        part.rename(columns={text_col: "text"}, inplace=True)

    df_neu["label"] = 0
    df_neg["label"] = 1
    df_pos["label"] = 2

    data = pd.concat(
        [
            df_neu[["text", "label"]],
            df_neg[["text", "label"]],
            df_pos[["text", "label"]]
        ],
        ignore_index=True
    )

    data["text"] = data["text"].fillna("").astype(str).str.strip()
    data = data[data["text"].str.len() >= 20]
    data = data.drop_duplicates(subset=["text"])

    return data.reset_index(drop=True)


if os.path.exists(DATA_PATH):
    df = load_dataset_from_rating_file(DATA_PATH)
    print("Данные загружены из общего файла с рейтингом:", DATA_PATH)
else:
    df = load_dataset_from_three_files(
        POSITIVE_PATH,
        NEUTRAL_PATH,
        NEGATIVE_PATH
    )
    print("Данные загружены из трех отдельных файлов.")

id2label = {
    0: "neutral",
    1: "negative",
    2: "positive"
}

label2id = {
    "neutral": 0,
    "negative": 1,
    "positive": 2
}

df["label_name"] = df["label"].map(id2label)

print("Размер датасета:", df.shape)
print(df["label_name"].value_counts())
df.head()

In [ ]:
TARGET_PER_CLASS = 1000

balanced_parts = []

for label in sorted(df["label"].unique()):
    part = df[df["label"] == label]

    if len(part) >= TARGET_PER_CLASS:
        part_balanced = resample(
            part,
            replace=False,
            n_samples=TARGET_PER_CLASS,
            random_state=RANDOM_STATE
        )
    else:
        part_balanced = resample(
            part,
            replace=True,
            n_samples=TARGET_PER_CLASS,
            random_state=RANDOM_STATE
        )

    balanced_parts.append(part_balanced)

balanced_df = pd.concat(balanced_parts)
balanced_df = balanced_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
balanced_df["label_name"] = balanced_df["label"].map(id2label)

print("Размер после балансировки:", balanced_df.shape)
print(balanced_df["label_name"].value_counts())

In [ ]:
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=balanced_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

print("\nTrain:")
print(train_df["label_name"].value_counts())

print("\nValidation:")
print(val_df["label_name"].value_counts())

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


train_dataset = ReviewsDataset(train_df, tokenizer)
val_dataset = ReviewsDataset(val_df, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

print("Базовая модель:", MODEL_NAME)
print("Количество классов:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1_weighted": f1
    }


common_training_args = dict(
    output_dir="./bert_reviews_cls_3class",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    seed=RANDOM_STATE,
    fp16=torch.cuda.is_available(),
)

try:
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        **common_training_args
    )
except TypeError:
    training_args = TrainingArguments(
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        **common_training_args
    )


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print("Eval:", eval_results)

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print("Accuracy:", round(accuracy_score(predictions.label_ids, preds), 4))
print("F1-score:", round(f1_score(predictions.label_ids, preds, average="weighted"), 4))

print(
    classification_report(
        predictions.label_ids,
        preds,
        target_names=["neutral", "negative", "positive"],
        digits=4
    )
)

In [ ]:
cm = confusion_matrix(predictions.label_ids, preds)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["neutral", "negative", "positive"],
    yticklabels=["neutral", "negative", "positive"]
)

plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Матрица ошибок модели BERT")
plt.show()

In [ ]:
!pip install -U shap

In [ ]:
import shap

model.to(device)
model.eval()


def predict_pos(texts):
    """
    Возвращает вероятность позитивного класса P(class=2).
    Именно этот класс используется для SHAP-интерпретации.
    """
    if isinstance(texts, str):
        texts = [texts]

    processed_texts = []

    for text in texts:
        if isinstance(text, list):
            processed_texts.append(" ".join(str(token) for token in text))
        else:
            processed_texts.append(str(text))

    enc = tokenizer(
        processed_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

    return probs[:, 2]


masker = shap.maskers.Text(tokenizer=r"\W+")

explainer = shap.Explainer(
    predict_pos,
    masker=masker
)

sample_texts = val_df["text"].sample(5, random_state=RANDOM_STATE).tolist()
shap_values = explainer(sample_texts)

shap.plots.text(shap_values[0])

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/bert_reviews_model_3class"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Сохранено в:", SAVE_DIR)

In [ ]:
config = AutoConfig.from_pretrained(SAVE_DIR)

print("num_labels:", config.num_labels)
print("id2label:", config.id2label)
print("label2id:", config.label2id)

In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

loaded_model.to(device)
loaded_model.eval()


def predict_sentiment(text):
    encoded = loaded_tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = loaded_model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    pred = int(np.argmax(probs))

    return {
        "text": text,
        "class_id": pred,
        "class_name": loaded_model.config.id2label[pred],
        "confidence": round(float(probs[pred]), 4),
        "neutral": round(float(probs[0]), 4),
        "negative": round(float(probs[1]), 4),
        "positive": round(float(probs[2]), 4),
    }


examples = [
    "Отличный товар, качество понравилось, буду заказывать еще.",
    "Обычный товар, ничего особенного, пользоваться можно.",
    "Пришел брак, качество плохое, больше не куплю."
]

for example in examples:
    print(predict_sentiment(example))